# 2. Transform it

Machine learning gave Python the derivative of anything written *inside a framework*. The header from notebook 1 is not inside a framework. A tracer cannot enter a compiled function. **Clad**, a Clang plugin, reads the function's own source and writes its derivative as C++ source.

At home there is one catch: the Clad plugin must match the Clang inside `cppjit` (21.1.8), and no package ships that pairing yet. So this notebook does what the talk's slide 6 does: it shows the C++ source Clad wrote for this header, verbatim, and compiles that in the session. The file says how it was generated. If you have the plugin, the last cell regenerates it live.

In [1]:
import cppjit, numpy as np, hashlib
cppjit.add_include_path("kernel"); cppjit.include("logistic_step.h")
print(open("kernel/logistic_step.h").read().split("#ifndef")[1][:200])   # the function being differentiated

 SC26_KERNEL_LOGISTIC_STEP_H
#define SC26_KERNEL_LOGISTIC_STEP_H

template <typename T>
T logistic_step(T N, T r, T K) {
    T growth = r * N * (T(1) - N / K);
    return N + growth;
}

#endif // SC26


## Why the tracer cannot do it

Try the usual thing. If JAX is installed, `jax.grad` fails on the compiled call: it sees an opaque Python callable, not a traceable computation. (Without JAX, as in the browser, the cell only says so.)

In [2]:
try:
    import jax, jax.numpy as jnp
    def step_jax(N, r, K): return N + r * N * (1 - N / K)             # the law, rewritten in JAX: fine
    print("JAX on its own copy of the law: d/dr =", jax.grad(step_jax, argnums=1)(50.0, 0.3, 2000.0))
    try:
        jax.grad(lambda r: cppjit.gbl.logistic_step["double"](50.0, r, 2000.0))(0.3)
    except Exception as e:
        print("JAX on the C++ as written:", type(e).__name__, "-- the tracer cannot enter a compiled function; it has no graph to differentiate")
except ImportError:
    print("No JAX here, as expected in the browser: a tracer needs the law rewritten in its own language")

No JAX here, as expected in the browser: a tracer needs the law rewritten in its own language


## The derivative Clad wrote

Clad lives inside this kernel's compiler. Two requests (`kernel/derivatives.py`) make it write two functions from `logistic_step<double>`, in the session: the forward-mode derivative with respect to `r`, and the reverse-mode gradient with respect to all three arguments. Read what it wrote: it is the chain rule applied line by line to the header's body.

In [3]:
from kernel import derivatives
print(derivatives.code("logistic_step"))         # the source Clad generated, from the session

double logistic_step_darg1(double N, double r, double K) {
    double _d_N = 0;
    double _d_r = 1;
    double _d_K = 0;
    double _t0 = r * N;
    double _t1 = (double(1) - N / K);
    double _d_growth = (_d_r * N + r * _d_N) * _t1 + _t0 * (0. - ((_d_N * K - N * _d_K) / (K * K)));
    double growth = _t0 * _t1;
    return _d_N + _d_growth;
}

void logistic_step_grad(double N, double r, double K, double *_d_N, double *_d_r, double *_d_K) {
    double _d_growth = 0.;
    double growth = r * N * (double(1) - N / K);
    {
        *_d_N += 1;
        _d_growth += 1;
    }
    {
        *_d_r += _d_growth * (double(1) - N / K) * N;
        *_d_N += r * _d_growth * (double(1) - N / K);
        *_d_N += -r * N * _d_growth / K;
        double _r0 = -r * N * _d_growth * -(N / (K * K));
        *_d_K += _r0;
    }
}



[clad] logistic_step_darg1, logistic_step_grad: generated by the Clad inside this kernel


In [4]:
derivatives.load("logistic_step")                # the two functions, under clad_generated
g = cppjit.gbl.clad_generated
N, r, K = 50.0, 0.30, 2000.0
print("Clad, forward mode, d/dr :", g.logistic_step_darg1(N, r, K))
dN, dr, dK = (np.zeros(1) for _ in range(3)); g.logistic_step_grad(N, r, K, dN, dr, dK)
print("Clad, reverse mode, grad :", (dN[0], dr[0], dK[0]))
print("closed form             :", (1 + r * (1 - 2 * N / K), N * (1 - N / K), r * N * N / K**2))

Clad, forward mode, d/dr : 48.75
Clad, reverse mode, grad : (np.float64(1.285), np.float64(48.75), np.float64(0.0001875))
closed form             : (1.285, 48.75, 0.0001875)


## "Wrong, not slow"

Finite differences are the alternative that needs no compiler. They are not slow here; they are *uncertain*. The error depends on a step size nobody knows in advance, and the best step moves with the function. Clad's number is the derivative of the code, to rounding.

In [5]:
f = lambda rr: cppjit.gbl.logistic_step["double"](N, rr, K)
exact = N * (1 - N / K)
print(f"{'step h':>10s} {'central FD':>18s} {'error':>10s}")
for h in (1e-1, 1e-2, 1e-4, 1e-6, 1e-8, 1e-10, 1e-12):
    fd = (f(r + h) - f(r - h)) / (2 * h); print(f"{h:10.0e} {fd:18.12f} {abs(fd - exact):10.1e}")
print(f"{'Clad':>10s} {g.logistic_step_darg1(N, r, K):18.12f} {abs(g.logistic_step_darg1(N, r, K) - exact):10.1e}")

    step h         central FD      error
     1e-01    48.750000000000    0.0e+00
     1e-02    48.750000000000    2.8e-13
     1e-04    48.749999999984    1.6e-11
     1e-06    48.749999997710    2.3e-09
     1e-08    48.750000303244    3.0e-07
     1e-10    48.750052883406    5.3e-05
     1e-12    48.743231673143    6.8e-03
      Clad    48.750000000000    0.0e+00


## If you have the plugin

With Clad built against the same Clang as cppjit, the derivative is generated in the session: three lines, as the talk's slide 8 says. Set `CLAD_PLUGIN` to the plugin's path and `CLAD_INCLUDE` to its include directory **before** importing cppjit (so this cell only works in a fresh kernel started with those variables). Otherwise, the quickest way to watch Clad write a derivative is Compiler Explorer: paste the header at godbolt.org and add `-fplugin=libclad.so`.

In [6]:
import os
if os.environ.get("CLAD_PLUGIN"):
    cppjit.add_include_path(os.environ["CLAD_INCLUDE"]); cppjit.include("clad/Differentiator/Differentiator.h")
    cppjit.cppdef('auto live = clad::differentiate(logistic_step<double>, "r");')
    print(cppjit.gbl.live.execute(50.0, 0.30, 2000.0)); print(cppjit.gbl.live.getCode())
else:
    print("no CLAD_PLUGIN in the environment; the generated source above was produced exactly this way (see its header comment)")

no CLAD_PLUGIN in the environment; the generated source above was produced exactly this way (see its header comment)


Next: a simulator with a loop, a clamp and a branch, and a fit through it, `3_fit_it.ipynb`.